# Stage 6: Prior-Knowledge Embedding Construction

> **v2 note.** The Stage 6 files described below are the ones built in March 2026; v2 did not rebuild
> them. What changed is how the model's initialization uses them (`rat_model_init.py`,
> `rat_load_prior_embedding.py`; v2 init job 11910364, 2026-10-08; the command is in
> `data/models/rat_genecompass_init_v2/provenance_rat_model_init.json`):
> - Tokens with a GeneCompass token keep GeneCompass's pre-trained prior rows; Stage 6 vectors are never
>   written there.
> - The T3 tokens copy their ortholog's pre-trained rows (T3a from the human ortholog, T3b from the mouse
>   ortholog) for all four knowledge types (`--t3-priors ortholog`).
> - Only the T4 tokens read Stage 6, and only for gene family and GRN (`--t4-zero-priors promoter,coexp`).
>   Promoter and co-expression stay zero by decision: the DNABert promoter vectors are not in GeneCompass's
>   coordinates, and neither the co-expression gene2vec nor an anchored rebuild on the cleaned corpus
>   recovered held-out genes' GeneCompass rows (pre-registered test, 2026-10-06).
> - How many T4 tokens get a non-zero family or GRN vector is counted from the Stage 6 files in the summary
>   cell below (it matches the v2 init log); that is all Stage 6 contributes to v2.
>
> April's initialization took every new token's priors from Stage 6 by rat ID, which left all T3a rows zero
> (fixed 2026-10-05). April's version of this notebook is in git history (commit `ad72cfa`).

## Stage Objective

Construct four complementary gene-level representations for co-expression, gene family, promoter sequence, and regulatory-network context.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Co-expression | Derive high-correlation gene pairs from the rat single-cell corpus and train a gene2vec model. |
| 2 | Gene family | Transfer HGNC family structure to mouse and rat orthologs and train a family-context gene2vec model. |
| 3 | Promoter sequence | Encode rat transcription-start-site windows with DNABert-6. |
| 4 | Regulatory network | Transfer human or mouse PECA2vec representations using the Stage 3 mapping hierarchy. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stages 2, 3, and 5 outputs plus HGNC, BioMart, genome-sequence, and GeneCompass prior-knowledge resources. |
| Primary outputs | Four embedding dictionaries in `data/training/prior_knowledge/`, each with the model's hidden size of features per represented gene. |

## Result Overview

Coverage differs by knowledge source and is reported explicitly; missing prior knowledge is represented by the documented zero-vector policy.

The sections below present the computational evidence in execution order. Every figure is printed by a code cell from the recorded stage outputs or the config; the markdown describes methods and points to those cells rather than quoting numbers.

In [1]:
import json, os
import numpy as np

PK_DIR = '../data/training/prior_knowledge'
MAN = {s: json.load(open(f'{PK_DIR}/stage6_{s}_manifest.json')) for s in ('coexp', 'family', 'promoter', 'grn')}
outputs = {
    'coexp_embeddings.pkl':       'Step 1 — co-expression gene2vec',
    'coexp_gene_pairs.txt':       f"Step 1 — {MAN['coexp']['outputs']['unique_pairs'] / 1e6:.1f}M unique pairs",
    'family_embeddings.pkl':      'Step 2 — gene family gene2vec',
    'family_gene_pairs.txt':      f"Step 2 — {MAN['family']['outputs']['gene_pairs'] / 1e6:.1f}M family pairs",
    'promoter_embeddings.pkl':    'Step 3 — DNABert CLS embeddings',
    'promoter_sequences.fa':      'Step 3 — extracted TSS sequences',
    'promoter_skipped.tsv':       'Step 3 — genes skipped + reason',
    'grn_embeddings.pkl':         'Step 4 — GRN PECA2vec transfer',
    'grn_transfer_coverage.tsv':  'Step 4 — per-gene source',
}

print(f"{'File':<35} {'On-disk':>8}  Description")
print('-' * 75)
for fname, desc in outputs.items():
    exists = '✓' if os.path.exists(f'{PK_DIR}/{fname}') else '✗'
    real_size = ''
    p = f'{PK_DIR}/{fname}'
    if os.path.exists(p):
        b = os.path.getsize(p)
        real_size = f'{b/1e6:.0f}M' if b > 1e6 else f'{b/1e3:.0f}K'
    print(f"{exists} {fname:<33} {real_size:>7}   {desc}")


File                                 On-disk  Description
---------------------------------------------------------------------------
✓ coexp_embeddings.pkl                  53M   Step 1 — co-expression gene2vec
✓ coexp_gene_pairs.txt                1202M   Step 1 — 36.4M unique pairs
✓ family_embeddings.pkl                 81M   Step 2 — gene family gene2vec
✓ family_gene_pairs.txt                284M   Step 2 — 8.9M family pairs
✓ promoter_embeddings.pkl               49M   Step 3 — DNABert CLS embeddings
✓ promoter_sequences.fa                 41M   Step 3 — extracted TSS sequences
✓ promoter_skipped.tsv                 236K   Step 3 — genes skipped + reason
✓ grn_embeddings.pkl                    67M   Step 4 — GRN PECA2vec transfer
✓ grn_transfer_coverage.tsv              2M   Step 4 — per-gene source


## Step 1 — Co-expression Embedding

**Script:** `pipeline/06_prior_knowledge/build_coexp_embedding.py` (parameters, counts and elapsed time are
printed from `stage6_coexp_manifest.json` by the next cell)

### Algorithm

1. Glob every quality-controlled h5ad file from Stage 2
2. Per file: sample **min(n_cells, `n_sample_cells`)** cells (GeneCompass paper value)
3. Pre-filter: keep genes with expression ratio ≥ `min_expr_ratio` (methodological addition)
4. Compute **nonzero PCC** for all gene pairs (both genes expressed ≥ 1 in shared cells)
5. Retain pairs where PCC > `pcc_threshold` and ≥ `min_shared_cells` shared cells
6. **Unify gene IDs** → human Ensembl via `rat_to_human_mapping.pickle` (T4 keeps ENSRNOG)
7. Deduplicate pairs across all files
8. Train **gene2vec** Skip-Gram model (`vector_size`, `window`, `epochs`, `workers`)

### Methodological additions beyond GeneCompass

| Addition | Rationale |
|----------|-----------|
| `min_shared_cells` | Prevent spurious PCC from 1–2 co-expressed cells |
| `min_expr_ratio` | Pre-filter rarely-expressed genes → reduce O(n²) compute |

Both additions are conservative and maintain the spirit of the GeneCompass approach.

In [2]:
with open(f'{PK_DIR}/stage6_coexp_manifest.json') as f:
    mf = json.load(f)
outs = mf['outputs']
print(f"Files processed:      {outs['files_processed']:,}")
print(f"Files skipped:        {outs['files_skipped']:,}")
print(f"Raw pairs (PCC>0.8):  {outs['raw_pairs_total']:,}")
print(f"Unique pairs:         {outs['unique_pairs']:,}")
print(f"Genes embedded:       {outs['embeddings']:,}")
print(f"Embedding dimension:  {outs['embedding_dimension']}")
print(f"Elapsed:              {mf['elapsed_seconds']/3600:.1f} h")
print(f"Parameters:           {mf['parameters']}")

Files processed:      864
Files skipped:        0
Raw pairs (PCC>0.8):  189,992,325
Unique pairs:         36,396,244
Genes embedded:       16,939
Embedding dimension:  768
Elapsed:              49.0 h
Parameters:           {'pcc_threshold': 0.8, 'n_sample_cells': 3000, 'min_shared_cells': 10, 'min_expr_ratio': 0.05, 'chunk_size': 2000, 'vector_size': 768, 'window': 1, 'sg': 1, 'epochs': 30, 'workers': 8}


## Step 2 — Gene Family Embedding

**Script:** `pipeline/06_prior_knowledge/build_family_embedding.py` (counts, parameters and elapsed time
printed from `stage6_family_manifest.json` by the next cell)

### Algorithm

1. Load HGNC complete set (auto-downloaded if absent) → human gene families
2. Derive mouse families via `mouse_human_orthologs.tsv`
3. Derive rat families via `rat_to_human_mapping.pickle` (set-valued inversion)
4. Generate all **within-family pairwise gene pairs**
5. Unify gene IDs → human Ensembl (T4 new-token genes keep ENSRNOG)
6. Train **gene2vec** Skip-Gram (same settings as Step 1)

### T4 gene coverage

T4 new rat tokens have **no HGNC family** membership because they lack human orthologs, so they carry a
**zero family embedding** — consistent with GeneCompass's handling of genes absent from prior knowledge
databases. The next cell prints the T4 count and how many of them were embedded.

In [3]:
with open(f'{PK_DIR}/stage6_family_manifest.json') as f:
    mf = json.load(f)
outs = mf['outputs']
print(f"Human gene families:  {outs['human_families']:,}")
print(f"Mouse gene families:  {outs['mouse_families']:,}")
print(f"Rat gene families:    {outs['rat_families']:,}")
print(f"Within-family pairs:  {outs['gene_pairs']:,}")
print(f"Genes embedded:       {outs['embeddings']:,}")
print(f"T4 genes:             {outs['t4_genes_total']:,}  ({outs['t4_genes_embedded']:,} with a family vector)")
print(f"Parameters:           {mf['parameters']}")
print(f"Embedding dimension:  {outs['embedding_dimension']}")
print(f"Elapsed:              {mf['elapsed_seconds']:.0f} s  ({mf['elapsed_seconds']/60:.1f} min)")
print()
print("Input checksums (MD5):")
for k, v in mf['inputs'].items():
    if isinstance(v, dict):
        print(f"  {k:<30}: {v.get('md5','N/A')}")

Human gene families:  1,927
Mouse gene families:  1,731
Rat gene families:    1,583
Within-family pairs:  8,864,659
Genes embedded:       26,019
T4 genes:             2,474  (0 with a family vector)
Parameters:           {'min_family_size': 2, 'vector_size': 768, 'window': 1, 'sg': 1, 'epochs': 30, 'workers': 8}
Embedding dimension:  768
Elapsed:              383 s  (6.4 min)

Input checksums (MD5):
  rat_token_mapping             : 26a810edf77c7c4b77cf22e9b70db266
  rat_to_human_mapping          : 317902827ed0012ad7c7622e1958fecc
  mouse_human_orthologs         : 4b03df71fe88430e49c4c53a493c2ba1
  hgnc_gene_families            : 5e1e18fea6f3b34dca80758c5322a9e6


In [4]:
import json
import matplotlib.pyplot as plt

with open(f'{PK_DIR}/stage6_family_manifest.json') as f:
    family_mf = json.load(f)
o = family_mf['outputs']
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
fig.suptitle('Stage 6: Gene Family Embedding', fontsize=13, fontweight='bold')
labels = ['Human', 'Mouse', 'Rat']
values = [o['human_families'], o['mouse_families'], o['rat_families']]
bars = axes[0].bar(labels, values, color=['#1976D2', '#00897B', '#F57C00'])
axes[0].bar_label(bars, labels=[f'{bar.get_height():,.0f}' for bar in bars], padding=3); axes[0].set_ylabel('Gene families'); axes[0].set_title('Families by species')
metrics = [o['gene_pairs'] / 1e6, o['embeddings'] / 1e3]
bars = axes[1].bar(['Gene pairs\n(millions)', 'Embedded genes\n(thousands)'], metrics, color=['#7B1FA2', '#C2185B'])
axes[1].bar_label(bars, fmt='%.1f'); axes[1].set_title('Training corpus and coverage')
for ax in axes: ax.spines[['top', 'right']].set_visible(False)
plt.show()


## Step 3 — Promoter Sequence Embedding (DNABert)

**Script:** `pipeline/06_prior_knowledge/build_promoter_embedding.py` (parameters, coverage and elapsed time
printed from `stage6_promoter_manifest.json` by the next cell)

### Algorithm

1. Retrieve TSS coordinates for every rat gene in the universe (Ensembl BioMart cache; release as set in the
   config `biomart` section)
2. Extract a window of `window_upstream` bp upstream and `window_downstream` bp downstream of the TSS
3. Skip genes where the N-masked fraction ≥ `max_n_fraction` (low-quality genome region)
4. Tokenize sequences as overlapping k-mers (`kmer`)
5. Run **DNABert-6** (`dnabert_model`) → take the **[CLS] token** as the embedding
6. Genes without TSS or with high-N sequence → **zero promoter embedding**

### Coverage

The next cell prints the extracted, embedded and skipped counts with the skip reasons, and which Stage 3
tiers and biotypes the skipped genes come from.

In [5]:
import csv
with open(f'{PK_DIR}/stage6_promoter_manifest.json') as f:
    mf = json.load(f)
outs = mf['outputs']
params = mf['parameters']
print(f"DNABert model:        {params['dnabert_model']}")
print(f"Window (bp):          -{params['window_upstream']}/+{params['window_downstream']} = {params['window_total_bp']:,} bp")
print(f"Max N fraction:       {params['max_n_fraction']}")
print(f"Batch size:           {params['batch_size']}")
print(f"Embedding dimension:  {outs['embedding_dimension']}")
print(f"Genes input:          {outs['genes_input']:,}")
print(f"Sequences extracted:  {outs['sequences_extracted']:,}")
print(f"Genes embedded:       {outs['embeddings']:,}")
print(f"Genes skipped:        {outs['genes_skipped']:,}")
print(f"Elapsed:              {mf['elapsed_seconds']:.0f} s  ({mf['elapsed_seconds']/60:.1f} min)")
print()
with open(f'{PK_DIR}/promoter_skipped.tsv') as f:
    skip_rows = list(csv.DictReader(f, delimiter='\t'))
reasons = {}
for r in skip_rows:
    reason = r['reason'].split(':')[0]
    reasons[reason] = reasons.get(reason, 0) + 1
print("Skip reasons:")
for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
    print(f"  {reason:<25}: {count:,}")

# which tiers / biotypes the skipped genes come from
import pandas as pd
_skip = pd.read_csv(f'{PK_DIR}/promoter_skipped.tsv', sep='\t')
_skip['reason'] = _skip['reason'].str.split(':').str[0]
_tm = pd.read_csv('../data/training/ortholog_mappings/rat_token_mapping.tsv', sep='\t', usecols=['rat_gene', 'tier', 'biotype'])
_skip = _skip.merge(_tm, on='rat_gene', how='left')
print("\nSkipped genes by reason × tier:")
print(pd.crosstab(_skip['reason'], _skip['tier']).to_string())
print("\nSkipped genes by reason × biotype:")
print(pd.crosstab(_skip['reason'], _skip['biotype']).to_string())

DNABert model:        zhihan1996/DNA_bert_6
Window (bp):          -500/+2000 = 2,500 bp
Max N fraction:       0.1
Batch size:           32
Embedding dimension:  768
Genes input:          22,213
Sequences extracted:  16,120
Genes embedded:       15,610
Genes skipped:        6,093
Elapsed:              409 s  (6.8 min)

Skip reasons:
  no_tss_coordinate        : 5,000
  high_n_fraction          : 1,090
  chrom_not_found          : 3



Skipped genes by reason × tier:
tier               T1_tri_species  T2a_human_one2one  T2b_mouse_one2one  T3a_human_multi  T3b_mouse_multi  T4_new_token
reason                                                                                                                 
chrom_not_found                 0                  0                  0                0                1             2
high_n_fraction               652                 36                142               71               29           160
no_tss_coordinate            3370                135                802              248              143           302

Skipped genes by reason × biotype:
biotype            lncrna  mirna  protein_coding
reason                                          
chrom_not_found         2      0               1
high_n_fraction        92     18             980
no_tss_coordinate     146      2            4852


In [6]:
import json
import matplotlib.pyplot as plt

with open(f'{PK_DIR}/stage6_promoter_manifest.json') as f:
    promoter_mf = json.load(f)
o, p = promoter_mf['outputs'], promoter_mf['parameters']
embedded = o['embeddings']; extracted_not_embedded = o['sequences_extracted'] - embedded; skipped = o['genes_skipped']
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
fig.suptitle('Stage 6: Promoter Sequence Embedding', fontsize=13, fontweight='bold')
bars = axes[0].bar(['Embedded', 'Sequence only', 'Skipped'], [embedded, extracted_not_embedded, skipped], color=['#00897B', '#FBC02D', '#D32F2F'])
axes[0].bar_label(bars, labels=[f'{bar.get_height():,.0f}' for bar in bars], padding=3); axes[0].set_ylabel('Genes'); axes[0].set_title('Promoter processing status')
axes[1].axis('off')
params = [['Model', p['dnabert_model']], ['Window', f"-{p['window_upstream']}/+{p['window_downstream']} bp"], ['k-mer', p['kmer']], ['Max N fraction', p['max_n_fraction']], ['Batch size', p['batch_size']], ['Embedding dim', o['embedding_dimension']]]
table = axes[1].table(cellText=params, colLabels=['Parameter', 'Value'], loc='center', cellLoc='left')
table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1, 1.45); axes[1].set_title('DNABert parameters')
plt.show()


## Step 4 — GRN Embedding (PECA2vec Transfer)

**Script:** `pipeline/06_prior_knowledge/build_grn_transfer.py` (a dictionary lookup, no training; elapsed
time printed below)

### Method

GeneCompass pre-trained **PECA2vec** embeddings on ENCODE chromatin-accessibility GRNs of human and mouse
cell types (GeneCompass paper). Rather than re-training GRN embeddings from scratch, we **transfer** directly:

1. For each rat gene, look up its unified human Ensembl ID (`rat_to_human_mapping`)
2. If found in human PECA2vec → use **human GRN embedding** (T1, T2a, T3a)
3. Else look up via mouse Ensembl ID → use **mouse GRN embedding** (T2b, T3b)
4. Otherwise (all T4 genes, and orthologs absent from PECA2vec) → **zero vector**

**MoTrPAC ATAC-seq intentionally excluded** to prevent data leakage between prior knowledge and Aim 3 endpoints.

### Coverage

The next cell prints how many genes take a human or a mouse PECA2vec vector, and how many get a zero vector.

In [7]:
with open(f'{PK_DIR}/stage6_grn_manifest.json') as f:
    mf = json.load(f)
outs = mf['outputs']
print("Method:", mf['method'])
print("Description:", mf['description'][:100] + "...")
print()
print(f"Genes total:       {outs['genes_total']:,}")
_n = outs['genes_total']
print(f"Human PECA2vec:    {outs['human_peca2vec']:,}  ({100 * outs['human_peca2vec'] / _n:.1f}%)")
print(f"Mouse PECA2vec:    {outs['mouse_peca2vec']:,}  ({100 * outs['mouse_peca2vec'] / _n:.1f}%)")
print(f"Zero vector:       {outs['zero_vector']:,}  ({100 * outs['zero_vector'] / _n:.1f}%; all T4 genes and orthologs absent from PECA2vec)")
print(f"Embedding dim:     {outs['embedding_dimension']}")
print(f"Elapsed:           {mf['elapsed_seconds']:.1f} s")

Method: transfer
Description: Cross-species GRN embedding transfer from GeneCompass PECA2vec. Human PECA2vec used as primary sourc...

Genes total:       22,213
Human PECA2vec:    15,234  (68.6%)
Mouse PECA2vec:    3,516  (15.8%)
Zero vector:       3,463  (15.6%; all T4 genes and orthologs absent from PECA2vec)
Embedding dim:     768
Elapsed:           1.4 s


In [8]:
import json
import matplotlib.pyplot as plt

def load_manifest(name):
    with open(f'{PK_DIR}/{name}') as f:
        return json.load(f)
coexp = load_manifest('stage6_coexp_manifest.json')['outputs']
family = load_manifest('stage6_family_manifest.json')['outputs']
promoter = load_manifest('stage6_promoter_manifest.json')['outputs']
grn = load_manifest('stage6_grn_manifest.json')['outputs']
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
fig.suptitle('Stage 6: Prior-Knowledge Coverage', fontsize=13, fontweight='bold')
grn_values = [grn['human_peca2vec'], grn['mouse_peca2vec'], grn['zero_vector']]
bars = axes[0].bar(['Human PECA2vec', 'Mouse PECA2vec', 'Zero vector'], grn_values, color=['#1976D2', '#00897B', '#9E9E9E'])
axes[0].bar_label(bars, labels=[f'{bar.get_height():,.0f}' for bar in bars], padding=3); axes[0].set_ylabel('Rat genes'); axes[0].set_title('GRN embedding source')
total = grn['genes_total']
coverage = [coexp['embeddings'], min(family['embeddings'], total), promoter['embeddings'], total]
bars = axes[1].bar(['Co-expression', 'Gene family', 'Promoter', 'GRN'], [100*x/total for x in coverage], color=['#7B1FA2', '#F57C00', '#C2185B', '#388E3C'])
axes[1].bar_label(bars, fmt='%.1f%%'); axes[1].set(ylabel='Coverage of rat gene universe (%)', ylim=(0, 125)); axes[1].set_title(f'Available vector coverage of {total:,} rat genes\n(gene-family coverage capped at 100%)')
for ax in axes: ax.spines[['top', 'right']].set_visible(False)
plt.show()


In [9]:
print("=" * 65)
print("STAGE 6 SUMMARY — Prior Knowledge Embeddings")
print("=" * 65)
n_univ = MAN['grn']['outputs']['genes_total']
print(f"  Rat gene universe:       {n_univ:,} genes (from Stage 3)")
print()
def _elapsed(s):
    return f"{s / 3600:.1f} h" if s >= 3600 else f"{s / 60:.1f} min" if s >= 60 else f"{s:.1f} s"
def _cov(n):   # the family model also embeds human/mouse family members, so it can exceed the universe
    return f"{n:,}/{n_univ:,} genes embedded"
grn_n = MAN['grn']['outputs']['human_peca2vec'] + MAN['grn']['outputs']['mouse_peca2vec']   # the rest get zero vectors
steps_summary = [
    ("Step 1 Co-expression", "gene2vec",  _cov(MAN['coexp']['outputs']['embeddings']),    _elapsed(MAN['coexp']['elapsed_seconds'])),
    ("Step 2 Gene family",   "gene2vec",  _cov(MAN['family']['outputs']['embeddings']),   _elapsed(MAN['family']['elapsed_seconds'])),
    ("Step 3 Promoter",      "DNABert-6", _cov(MAN['promoter']['outputs']['embeddings']), _elapsed(MAN['promoter']['elapsed_seconds'])),
    ("Step 4 GRN",           "PECA2vec",  _cov(grn_n) + f" (+{MAN['grn']['outputs']['zero_vector']:,} zero)",                                   _elapsed(MAN['grn']['elapsed_seconds'])),
]
print(f"  {' Step':<22} {' Model':<12} {' Coverage':<34} {' Elapsed'}")
print("  " + "-" * 78)
for name, model, cov, elapsed in steps_summary:
    print(f"  {name:<22} {model:<12} {cov:<34} {elapsed}")
print()
_dims = sorted({MAN[s]['outputs']['embedding_dimension'] for s in MAN})
_pkl = [f'{PK_DIR}/{s}_embeddings.pkl' for s in ('coexp', 'family', 'promoter', 'grn')]
print(f"  Embedding dimension: {', '.join(map(str, _dims))}")
print(f"  Output: {len(_pkl)} × .pkl files, total {sum(os.path.getsize(p) for p in _pkl) / 1e6:,.0f} MB")
print()
# what v2's initialization takes from Stage 6: T4 rows, family + GRN only (provenance_rat_model_init.json),
# looked up by rat ID; a row counts only if it is non-zero (the pickles hold explicit zero vectors)
import pickle, pandas as pd
_init = json.load(open('../data/models/rat_genecompass_init_v2/provenance_rat_model_init.json'))
_cmd = _init['command']
_zero = _cmd[_cmd.index('--t4-zero-priors') + 1].split(',')
_tm = pd.read_csv('../data/training/ortholog_mappings/rat_token_mapping.tsv', sep='\t', usecols=['rat_gene', 'tier'])
_t4 = _tm.loc[_tm['tier'] == 'T4_new_token', 'rat_gene'].tolist()
print("  Downstream (v2 init, rat_genecompass_init_v2):")
print(f"    T4 tokens only; priors left zero by --t4-zero-priors: {', '.join(_zero)}")
for _k in ('family', 'grn'):
    with open(f'{PK_DIR}/{_k}_embeddings.pkl', 'rb') as f:
        _e = pickle.load(f)
    _nz = sum(1 for g in _t4 if g in _e and np.any(np.asarray(_e[g]) != 0))
    print(f"    T4 {_k:<7} non-zero: {_nz:,} / {len(_t4):,}")

STAGE 6 SUMMARY — Prior Knowledge Embeddings
  Rat gene universe:       22,213 genes (from Stage 3)

   Step                   Model        Coverage                           Elapsed
  ------------------------------------------------------------------------------
  Step 1 Co-expression   gene2vec     16,939/22,213 genes embedded       49.0 h
  Step 2 Gene family     gene2vec     26,019/22,213 genes embedded       6.4 min
  Step 3 Promoter        DNABert-6    15,610/22,213 genes embedded       6.8 min
  Step 4 GRN             PECA2vec     18,750/22,213 genes embedded (+3,463 zero) 1.4 s

  Embedding dimension: 768
  Output: 4 × .pkl files, total 250 MB

  Downstream (v2 init, rat_genecompass_init_v2):
    T4 tokens only; priors left zero by --t4-zero-priors: promoter, coexp


    T4 family  non-zero: 0 / 2,474


    T4 grn     non-zero: 36 / 2,474


## Stage Summary

Stage 6 produces four prior-knowledge channels. The v2 model reads them only for its T4 tokens, family and
GRN only; the summary cell above counts the non-zero rows that leaves (see also the note at the top).
Coverage diagnostics distinguish observed embeddings from zero-vector fallbacks, and the manifests record
source checksums, model parameters, and known determinism limitations.

---

## Reproducibility Notes

### Environment

| Item | Value |
|------|-------|
| Cluster | Purdue Gilbreth (SLURM) |
| Python env | `motrpac-env` (venv) |
| Step 3 device | CPU |

### Run dates, configuration and input checksums

The next cell prints, from the four Stage 6 manifests, each step's generation time, elapsed time,
parameters and input checksums.

> Steps 2–4 completed before Step 1 (Step 1 ran with a higher PCC candidate set in an earlier
> iteration, then was re-run with `min_expr_ratio` to reduce compute).

### Determinism

- **Steps 1 & 2 (gene2vec)**: Skip-Gram training with several `workers` is **non-deterministic** (thread-level floating-point race). gensim is seeded, but multi-worker output may vary slightly between runs.
- **Step 3 (DNABert)**: Deterministic given fixed weights and sequences.
- **Step 4 (GRN transfer)**: Fully deterministic (dict lookup).

For publication: re-run Steps 1 & 2 with `workers=1` to guarantee bit-identical embeddings. (The deposited
pickles are what v2's initialization read; see `docs/repro_stages1_7/REPRODUCTION.md`.)

### Re-running Stage 6

```bash
cd ..

# Run all 4 steps (Step 1 takes about two days)
python pipeline/run_stage6.py

# Step 1 only (submit as long SLURM job)
sbatch --time=60:00:00 --mem=32G --cpus-per-task=8 \
       --wrap="python pipeline/run_stage6.py --from 1" \
       --job-name=stage6_coexp

# Steps 2–4 only (minutes)
python pipeline/run_stage6.py --from 2

# Validate without running
python pipeline/run_stage6.py --dry-run
```

### MoTrPAC data leakage safeguard

The GRN embedding step **explicitly excludes** MoTrPAC ATAC-seq data from the PECA2vec
transfer. Only the public ENCODE GRNs GeneCompass trained on are used as source.
This ensures the prior knowledge does not leak exercise-specific regulatory information
into Stage 7 fine-tuning, which would bias downstream Aim 2 (deconvolution) and
Aim 3 (cross-species translation) evaluations.

In [10]:
import platform
print(f"Python  {platform.python_version()}\n")
for _s in ('coexp', 'family', 'promoter', 'grn'):
    _m = MAN[_s]
    print(f"{_s:<9} generated {_m['generated'][:16]} UTC   elapsed {_elapsed(_m['elapsed_seconds'])}")
    if _m.get('parameters'):
        print(f"          parameters: {_m['parameters']}")
    for _name, _inp in (_m.get('inputs') or {}).items():
        if isinstance(_inp, dict) and 'md5' in _inp:
            print(f"          input md5  {_name:<24} {_inp['md5']}")

Python  3.12.11

coexp     generated 2026-03-20T17:30 UTC   elapsed 49.0 h
          parameters: {'pcc_threshold': 0.8, 'n_sample_cells': 3000, 'min_shared_cells': 10, 'min_expr_ratio': 0.05, 'chunk_size': 2000, 'vector_size': 768, 'window': 1, 'sg': 1, 'epochs': 30, 'workers': 8}
          input md5  rat_to_human_mapping     317902827ed0012ad7c7622e1958fecc
family    generated 2026-03-18T00:44 UTC   elapsed 6.4 min
          parameters: {'min_family_size': 2, 'vector_size': 768, 'window': 1, 'sg': 1, 'epochs': 30, 'workers': 8}
          input md5  rat_token_mapping        26a810edf77c7c4b77cf22e9b70db266
          input md5  rat_to_human_mapping     317902827ed0012ad7c7622e1958fecc
          input md5  mouse_human_orthologs    4b03df71fe88430e49c4c53a493c2ba1
          input md5  hgnc_gene_families       5e1e18fea6f3b34dca80758c5322a9e6
promoter  generated 2026-03-18T15:57 UTC   elapsed 6.8 min
          parameters: {'window_upstream': 500, 'window_downstream': 2000, 'window_total_bp

In [11]:
# Verify all 4 embedding outputs exist and are loadable (headers only)
import pickle, os

PK_DIR = '../data/training/prior_knowledge'
emb_files = [
    'coexp_embeddings.pkl',
    'family_embeddings.pkl',
    'promoter_embeddings.pkl',
    'grn_embeddings.pkl',
]
total_genes = sum(1 for _ in open('../data/training/gene_universe/pruned_gene_universe.tsv')) - 1
print("Embedding file verification")
print(f"{'File':<30} {'Size':>7}  {'N_keys':>9}  {'Dim':>5}  {'Status'}")
print('-' * 65)
for fname in emb_files:
    fpath = f'{PK_DIR}/{fname}'
    size_mb = os.path.getsize(fpath) / 1e6
    try:
        with open(fpath, 'rb') as f:
            emb = pickle.load(f)
        n_genes = len(emb)
        dim = len(next(iter(emb.values())))
        n_nz = sum(1 for v in emb.values() if np.any(np.asarray(v) != 0))
        print(f"{fname:<30} {size_mb:>6.0f}M  {n_genes:>9,}  {dim:>5}  ✓ ({n_nz:,} non-zero keys; universe {total_genes:,})")
    except Exception as e:
        print(f"{fname:<30}  ERROR: {e}")

Embedding file verification
File                              Size     N_keys    Dim  Status
-----------------------------------------------------------------


coexp_embeddings.pkl               53M     16,939    768  ✓ (16,939 non-zero keys; universe 22,213)
family_embeddings.pkl              81M     26,019    768  ✓ (26,019 non-zero keys; universe 22,213)


promoter_embeddings.pkl            49M     15,610    768  ✓ (15,610 non-zero keys; universe 22,213)
grn_embeddings.pkl                 67M     21,455    768  ✓ (17,992 non-zero keys; universe 22,213)
